<a href="https://colab.research.google.com/github/Akshatha7710/smart-tea-estate-management-system/blob/soil_quality-analysis_and_predictive_modeling/Soil_Quality_Analysis_and_Predictive_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ==========================================================
# Import Libraries
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, r2_score
from pmdarima import auto_arima

# ==========================================================
# Prepare Rainfall Data
# ==========================================================
rainfall = pd.read_csv("sample_data/rainfall_data.csv")
rainfall["Date"] = pd.to_datetime(rainfall["Date"])
rainfall = rainfall.sort_values("Date")

# Smooth rainfall slightly to reduce noise
rainfall["Rainfall_Smoothed"] = rainfall["Rainfall"].rolling(3).mean()

rain_series = rainfall.set_index("Date")["Rainfall_Smoothed"].dropna()

# Ensure monthly frequency
rain_series = rain_series.asfreq("MS")

# ==========================================================
# Train-Test Split
# ==========================================================

train_size = int(len(rain_series) * 0.8)

train = rain_series[:train_size]
test = rain_series[train_size:]

# ==========================================================
# Automatically Find Best SARIMA Parameters
# ==========================================================

auto_model = auto_arima(
    train,
    seasonal=True,
    m=12,
    trace=False,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

print("Best SARIMA Order:", auto_model.order)
print("Best Seasonal Order:", auto_model.seasonal_order)

# ==========================================================
# Train SARIMA Model
# ==========================================================

sarima_model = SARIMAX(
    train,
    order=auto_model.order,
    seasonal_order=auto_model.seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_results = sarima_model.fit()

# ==========================================================
# Predict Test Data
# ==========================================================

pred = sarima_results.get_forecast(steps=len(test))
pred_mean = pred.predicted_mean

# ==========================================================
# Model Evaluation
# ==========================================================

mae = mean_absolute_error(test, pred_mean)
r2 = r2_score(test, pred_mean)

print("\nRainfall Prediction Performance")
print("MAE:", mae)
print("R2:", r2)

# ==========================================================
# Forecast Next 12 Months
# ==========================================================

future_forecast = sarima_results.get_forecast(steps=12)

forecast_mean = future_forecast.predicted_mean
forecast_ci = future_forecast.conf_int()

print("\nRainfall Forecast (Next 12 Months)")
print(forecast_mean)

# ==========================================================
# Visualization
# ==========================================================

plt.figure(figsize=(12,6))

plt.plot(train.index, train, label="Training Data")
plt.plot(test.index, test, label="Actual Rainfall")
plt.plot(test.index, pred_mean, label="Predicted Rainfall")

future_dates = pd.date_range(
    start=rain_series.index[-1] + pd.DateOffset(months=1),
    periods=12,
    freq="MS"
)

plt.plot(future_dates, forecast_mean, label="Future Forecast")

plt.fill_between(
    future_dates,
    forecast_ci.iloc[:,0],
    forecast_ci.iloc[:,1],
    alpha=0.25
)

plt.title("Improved Rainfall Forecast (SARIMA)")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.legend()
plt.grid(True)

plt.show()

KeyError: 'Date'